## Script per fine-tuning
fine-tuning di cnn_dailymail filtrato opportunamente con tecniche di data analysis su bart-large (da eseguire du google collab)



In [ ]:
!pip install datasets transformers evaluate rouge_score numpy

In [ ]:
from datasets import load_dataset
# Preparazione del dataset come oggetto DatasetDict con sezioni per train, validation e test
ds = load_dataset("parquet", data_files="/content/cnn_filtered.parquet")
# 80% train, 20% test
ds = ds.train_test_split(test_size=0.2, seed=42)
train_val = ds["train"]
test = ds["test"]

# 10% validation 70% train
train_val = train_val.train_test_split(test_size=0.125, seed=42)
train = train_val["train"]
val = train_val["test"]

dataset_dict = DatasetDict({
    "train": train,
    "validation": val,
    "test": test
})


Generating train split: 100015 examples [00:04, 23545.29 examples/s]


In [ ]:
from datasets import DatasetDict
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM,
                          DataCollatorForSeq2Seq)
import evaluate
import numpy as np


model_name = "facebook/bart-large"   
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
model     = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Il modello accetta input fino a 1024 token
max_input_length  = 1024
max_target_length = 128   # tipico per CNN/DM; puoi adattare (o parametrizzare per %)

def preprocess(batch):
    # testo sorgente e target
    inputs  = batch["article_clean"]
    targets = batch["highlights"]
    model_inputs = tokenizer(
        inputs, max_length=max_input_length, truncation=True
    )
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            targets, max_length=max_target_length, truncation=True
        )["input_ids"]
    # Ignora il padding nella loss
    model_inputs["labels"] = [
        [(t if t != tokenizer.pad_token_id else -100) for t in seq] for seq in labels
    ]
    return model_inputs

# Oggetto DatsetDict tokenizzato
tokenized_ds = dataset_dict.map(
    preprocess,
    batched=True,
    remove_columns=dataset_dict["train"].column_names
)

# Data collator (gestisce padding dinamico)
collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# ROUGE per valutazione 
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    # Decodifica predizioni
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    # Sostituisci -100 con pad_token_id, poi decodifica labels
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = rouge.compute(
        predictions=[p.strip() for p in decoded_preds],
        references=[l.strip() for l in decoded_labels],
        use_stemmer=True
    )
    # Stampa i risultati rouge in percentuale
    result = {k: round(v * 100, 2) for k, v in result.items()}
    return result


Map:   0%|          | 0/90013 [00:00<?, ? examples/s]/root/NLP_sumarization/.venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:4007: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
Map: 100%|██████████| 5001/5001 [00:04<00:00, 1018.34 examples/s]


TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'predict_with_generate'

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
# TrainingArguments
args = Seq2SeqTrainingArguments(
    output_dir="bart_cnn_finetuned",
    per_device_train_batch_size=8,          
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=8,          
    learning_rate=5e-5,
    num_train_epochs=4,
    weight_decay=0.01,
    logging_steps=100,
    eval_strategy="epoch",
    eval_steps=1000,
    save_strategy="steps",
    save_steps=5000,
    save_total_limit=2,    
    bf16=True,                         
    report_to="none",
    predict_with_generate=True,
    generation_max_length=256,    # lunghezza max riassunto
    generation_num_beams=4, # il numero di sequenze candidate iniziali e tenute in parallelo
)

# Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    data_collator=collator,
    compute_metrics=compute_metrics,
)

# Train + Eval + Test
trainer.train()
eval_metrics = trainer.evaluate()
print("Validation:", eval_metrics)

# Test set (se presente)
if "test" in tokenized_ds:
    test_metrics = trainer.evaluate(eval_dataset=tokenized_ds["test"])
    print("Test:", test_metrics)

# Salva modello
trainer.save_model("bart_cnn_finetuned/final")
tokenizer.save_pretrained("bart_cnn_finetuned/final")
